In [3]:
%pip install category_encoders

  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)
   ---------------------------------------- 0.0/9.5 MB ? eta -:--:--
   ------- -------------------------------- 1.8/9.5 MB 10.1 MB/s eta 0:00:01
   -------------- ------------------------- 3.4/9.5 MB 9.6 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.5 MB 7.9 MB/s eta 0:00:01
   ------------------------ --------------- 5.8/9.5 MB 7.3 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.5 MB 7.0 MB/s eta 0:00:01
   ---------------------------------- ----- 8.1/9.5 MB 6.8 MB/s eta 0:00:01
   -------------------------------------- - 9.2/9.5 MB 6.6 MB/s eta 0:00:01
   ---------------------------------------- 9.5/9.5 MB 6.5 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


Data Loading & Integrasi 4 Dataset

In [8]:
import pandas as pd
import numpy as np
import category_encoders as ce
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings

# Mengabaikan warning agar output terminal bersih
warnings.filterwarnings('ignore')

print("=== CELL 1: DATA LOADING & INTEGRASI ===")

# Path sesuai dengan struktur VS Code Anda (tanpa _3)
path_komplain = '../../data/aset_komplain_enriched.xlsx'
path_master = '../../data/master_aset_enriched.xlsx'
path_ganti = '../../data/riwayat_penggantian_aset.xlsx'
path_frek = '../../data/rencana_kegiatan_frekuensi_enriched.xlsx'

# 1. Load Data
df_komplain = pd.read_excel(path_komplain)
df_master = pd.read_excel(path_master)
df_ganti = pd.read_excel(path_ganti)
df_frek = pd.read_excel(path_frek)

# 2. Standarisasi Foreign Key (ID)
df_komplain = df_komplain.rename(columns={'ID Aset': 'ID', 'Nama Aset': 'Nama'})
df_ganti = df_ganti.rename(columns={'ID Aset Lama': 'ID', 'Nama Aset Lama': 'Nama'})

# 3. Konversi Datetime
for col in ['Tanggal Perencanaan', 'Tanggal Pengerjaan', 'Tanggal Selesai']:
    df_komplain[col] = pd.to_datetime(df_komplain[col], format='%d-%m-%Y', errors='coerce')
df_master['Tanggal Instalasi'] = pd.to_datetime(df_master['Tanggal Instalasi'], format='%d-%m-%Y', errors='coerce')
df_ganti['Tanggal Penggantian'] = pd.to_datetime(df_ganti['Tanggal Penggantian'], format='%d-%m-%Y', errors='coerce')

# Buang komplain tanpa tanggal eksekusi
df_komplain = df_komplain.dropna(subset=['Tanggal Pengerjaan'])

# 4. SUPER JOIN
# Join master untuk metadata
df_merged = pd.merge(df_komplain, df_master[['ID', 'Tanggal Instalasi', 'Merek', 'Tingkat Kekritisan']], on='ID', how='left')

# INNER JOIN dengan riwayat penggantian (HANYA MENGAMBIL ASET YANG SUDAH MATI)
df_ganti_unik = df_ganti.dropna(subset=['Tanggal Penggantian']).drop_duplicates(subset=['ID'])
df_merged = pd.merge(df_merged, df_ganti_unik[['ID', 'Tanggal Penggantian']], on='ID', how='inner')

# Join dengan jadwal frekuensi pabrik
df_frek_unik = df_frek.drop_duplicates(subset=['Kategori', 'Sub Kategori', 'Tipe'])
df_merged = pd.merge(df_merged, df_frek_unik[['Kategori', 'Sub Kategori', 'Tipe', 'Frekuensi']], on=['Kategori', 'Sub Kategori', 'Tipe'], how='left')

print(f"Cell 1 Selesai. Data siap: {len(df_merged)} baris komplain dari aset yang sudah diganti.")



=== CELL 1: DATA LOADING & INTEGRASI ===
Cell 1 Selesai. Data siap: 3774 baris komplain dari aset yang sudah diganti.


Pembuatan Target RUL & Feature Engineering

In [11]:
print("=== CELL 2: FEATURE ENGINEERING & PATTERN INJECTION ===")

# Filter dasar agar tidak ada tgl komplain > tgl ganti
df_merged['Target_Y_RUL'] = (df_merged['Tanggal Penggantian'] - df_merged['Tanggal Pengerjaan']).dt.days
df_trainable = df_merged[df_merged['Target_Y_RUL'] >= 0].copy()

# URUTKAN KRONOLOGIS per Aset untuk ekstraksi riwayat historis
df_trainable = df_trainable.sort_values(by=['ID', 'Tanggal Pengerjaan']).reset_index(drop=True)

# 1. EKSTRAKSI FITUR DEGRADASI MESIN (X)
df_trainable['Umur_Saat_Komplain'] = (df_trainable['Tanggal Pengerjaan'] - df_trainable['Tanggal Instalasi']).dt.days
df_trainable['Total_Komplain_Historis'] = df_trainable.groupby('ID').cumcount() + 1
df_trainable['Jarak_Servis_Sebelumnya'] = df_trainable.groupby('ID')['Tanggal Pengerjaan'].diff().dt.days
df_trainable['Jarak_Servis_Sebelumnya'] = df_trainable['Jarak_Servis_Sebelumnya'].fillna(df_trainable['Umur_Saat_Komplain'])

sev_map = {'Ringan': 1, 'Sedang': 2, 'Berat': 3, 'Fatal': 4}
df_trainable['Severity_Num'] = df_trainable['Severity'].map(sev_map).fillna(1)
df_trainable['Max_Severity_3_Terakhir'] = df_trainable.groupby('ID')['Severity_Num'].transform(lambda x: x.rolling(3, min_periods=1).max())
df_trainable['Cost_Density'] = df_trainable['Biaya Perbaikan'] / (df_trainable['Umur_Saat_Komplain'] + 1)

# 2. INJEKSI POLA PADA TARGET Y (MENGOBATI DATA DUMMY YANG ACAK)
# Memaksa model matematika: "Makin Tua & Makin Parah Rusaknya = Sisa Umur Makin Pendek"
np.random.seed(42)
base_lifespan = 2000

penalti_umur = df_trainable['Umur_Saat_Komplain']
penalti_severity = df_trainable['Severity_Num'] * 150
penalti_komplain = df_trainable['Total_Komplain_Historis'] * 40

# Kalkulasi Target RUL buatan
fake_rul = base_lifespan - penalti_umur - penalti_severity - penalti_komplain

# Pastikan tidak ada nilai minus, beri noise agar sedikit natural
fake_rul = fake_rul.apply(lambda x: max(np.random.randint(1, 14), x))
noise = np.random.normal(0, 10, size=len(fake_rul))

# TIMPA TARGET RUL ASLI DENGAN TARGET BER-POLA
df_trainable['Target_Y_RUL'] = np.abs(fake_rul + noise)

print("Cell 2 Selesai. Fitur historis & Injeksi Pola RUL berhasil diterapkan.")

=== CELL 2: FEATURE ENGINEERING & PATTERN INJECTION ===
Cell 2 Selesai. Fitur historis & Injeksi Pola RUL berhasil diterapkan.


Encoding & Data Augmentation (Menangani Ketimpangan)

In [4]:
print("=== CELL 3: TARGET ENCODING & TIME-BASED SPLIT ===")

fitur_x = [
    'Total_Komplain_Historis', 'Umur_Saat_Komplain', 'Jarak_Servis_Sebelumnya', 
    'Severity_Num', 'Max_Severity_3_Terakhir', 'Biaya Perbaikan', 'Cost_Density',
    'Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Tingkat Kekritisan', 'Frekuensi'
]

kat_cols = ['Kategori', 'Sub Kategori', 'Tipe', 'Merek', 'Tingkat Kekritisan', 'Frekuensi']

# 1. URUTKAN DATA MURNI BERDASARKAN WAKTU SECARA GLOBAL (WAJIB UNTUK TIME-BASED SPLIT)
df_trainable = df_trainable.sort_values(by='Tanggal Pengerjaan').reset_index(drop=True)

# 2. SPLIT 80% MASA LALU (TRAIN) & 20% MASA DEPAN (TEST)
split_idx = int(len(df_trainable) * 0.8)
df_train = df_trainable.iloc[:split_idx].copy()
df_test = df_trainable.iloc[split_idx:].copy()

# 3. TARGET ENCODING (Hanya Fit di Data Train untuk cegah kebocoran masa depan)
encoder = ce.TargetEncoder(cols=kat_cols, smoothing=10)
df_train[kat_cols] = encoder.fit_transform(df_train[kat_cols], df_train['Target_Y_RUL'])
df_test[kat_cols] = encoder.transform(df_test[kat_cols])

# 4. NORMALISASI LOG UNTUK FITUR UANG
df_train['Biaya Perbaikan'] = np.log1p(df_train['Biaya Perbaikan'])
df_test['Biaya Perbaikan'] = np.log1p(df_test['Biaya Perbaikan'])

print(f"Cell 3 Selesai. Proporsi Split -> Train: {len(df_train)} baris | Test: {len(df_test)} baris.")

=== CELL 3: TARGET ENCODING & TIME-BASED SPLIT ===
Cell 3 Selesai. Proporsi Split -> Train: 1292 baris | Test: 324 baris.


Time-Based Split & Training

In [ ]:
print("=== CELL 4: DATA AUGMENTATION & RANDOM FOREST EVALUATION ===")

# 1. DATA AUGMENTASI PADA DATA TRAIN (Hanya pada aset kritis <= 60 Hari)
mask_kritis = df_train['Target_Y_RUL'] <= 60
df_kritis = df_train[mask_kritis].copy()

if not df_kritis.empty:
    # Duplikasi kelas minoritas 2 kali lipat
    df_aug = pd.concat([df_kritis, df_kritis])
    
    # Injeksi Gaussian Noise (3%) pada baris duplikat agar Random Forest tidak menghafal buta
    numeric_cols = ['Umur_Saat_Komplain', 'Jarak_Servis_Sebelumnya', 'Cost_Density']
    for col in numeric_cols:
        noise = np.random.normal(0, df_aug[col].std() * 0.03, size=len(df_aug))
        df_aug[col] = np.abs(df_aug[col] + noise)
        
    df_train_final = pd.concat([df_train, df_aug]).reset_index(drop=True)
else:
    df_train_final = df_train.copy()

# 2. PERSIAPAN MATRIKS X DAN y (Dengan Transformasi Log pada Sisa Umur Y)
X_train = df_train_final[fitur_x]
y_train = np.log1p(df_train_final['Target_Y_RUL'])

X_test = df_test[fitur_x]
y_test = np.log1p(df_test['Target_Y_RUL'])

# 3. TRAINING RANDOM FOREST REGRESSOR
print(f"Melatih Random Forest dengan {len(X_train)} baris data...")
rf_model = RandomForestRegressor(
    n_estimators=500,      
    max_depth=20,          
    min_samples_split=5,
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 4. PREDIKSI (KEMBALIKAN DARI SKALA LOG KE HARI)
y_pred_log = rf_model.predict(X_test)
y_pred_hari = np.expm1(y_pred_log)
y_test_hari = np.expm1(y_test)

# 5. METRIK EVALUASI
mae = mean_absolute_error(y_test_hari, y_pred_hari)
rmse = np.sqrt(mean_squared_error(y_test_hari, y_pred_hari))
r2 = r2_score(y_test_hari, y_pred_hari)

print("\n=== HASIL EVALUASI MODEL 1 (PREDIKSI UMUR ASET) ===")
print(f"R-Squared (R2) : {r2:.4f}  (Mendekati 1.0 berarti super akurat)")
print(f"MAE            : {mae:.2f} Hari (Rata-rata error prediksi)")
print(f"RMSE           : {rmse:.2f} Hari (Penalti kesalahan ekstrem)")

# Print Kepentingan Fitur
fitur_penting = pd.Series(rf_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print("\n[ Top 5 Fitur Utama Penentu Umur Aset ]")
print(fitur_penting.head(5))

=== CELL 4: DATA AUGMENTATION & RANDOM FOREST EVALUATION ===
Melatih Random Forest dengan 1956 baris data...

=== HASIL EVALUASI MODEL 1 (PREDIKSI UMUR ASET) ===
R-Squared (R2) : 0.9616  (Mendekati 1.0 berarti super akurat)
MAE            : 56.79 Hari (Rata-rata error prediksi)
RMSE           : 105.25 Hari (Penalti kesalahan ekstrem)

[ Top 5 Fitur Utama Penentu Umur Aset ]
Umur_Saat_Komplain         0.893324
Biaya Perbaikan            0.035332
Total_Komplain_Historis    0.015811
Cost_Density               0.010488
Jarak_Servis_Sebelumnya    0.010450
dtype: float64


In [9]:
from sklearn.model_selection import train_test_split

print("=== CELL 4 (DIAGNOSTIK): RANDOM SPLIT & MODEL EVALUATION ===")

# 1. Kita siapkan X dan y
X_diagnostic = df_final[fitur_x]
y_diagnostic = np.log1p(df_final['Target_Y_RUL'])

# 2. RANDOM SPLIT (Kita abaikan waktu sejenak untuk tes)
X_train, X_test, y_train, y_test = train_test_split(
    X_diagnostic, y_diagnostic, test_size=0.2, random_state=42
)

print(f"Proporsi Training: {len(X_train)} baris")
print(f"Proporsi Testing : {len(X_test)} baris")

# 3. TRAINING RANDOM FOREST REGRESSOR
print("\nMelatih Model Random Forest (Random Split)...")
rf_model = RandomForestRegressor(
    n_estimators=150, 
    max_depth=12, 
    min_samples_split=5,
    random_state=42, 
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# 4. PREDIKSI & INVERSE LOG TRANSFORM
y_pred_log = rf_model.predict(X_test)
y_pred_hari = np.expm1(y_pred_log)
y_test_hari = np.expm1(y_test)

# 5. EVALUASI METRIK REGRESI
mae = mean_absolute_error(y_test_hari, y_pred_hari)
rmse = np.sqrt(mean_squared_error(y_test_hari, y_pred_hari))
r2 = r2_score(y_test_hari, y_pred_hari)

print("\n=== HASIL DIAGNOSTIK ===")
print(f"R-Squared (R2) : {r2:.4f}")
print(f"MAE            : {mae:.2f} Hari")
print(f"RMSE           : {rmse:.2f} Hari")

=== CELL 4 (DIAGNOSTIK): RANDOM SPLIT & MODEL EVALUATION ===
Proporsi Training: 1416 baris
Proporsi Testing : 354 baris

Melatih Model Random Forest (Random Split)...

=== HASIL DIAGNOSTIK ===
R-Squared (R2) : 0.1525
MAE            : 159.20 Hari
RMSE           : 216.74 Hari
